<a href="https://colab.research.google.com/github/Lujain-Mahesar/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lujain-Mahesar/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found, check repo root"
print("Data found, ready to go.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Data found, ready to go.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane: figuring out which pages actually need a content refresh.

I'm calling this a ranking problem, not just plain classification. Here's why,
it's not enough to just tag a page as "declining" or "fine." What actually
matters is putting the pages in order, so the ones that need attention most
show up first. A content team can only realistically go through a handful of
pages a week, so ranking them by priority is what actually helps, not just
sorting them into two buckets.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
df_check = pd.read_csv("data/raw/content_refresh_anonymized.csv")
declining_count = (df_check["trend_direction"] == "down").sum()
print(f"{declining_count} pages are labeled 'declining' out of {df_check.shape[0]} total.")
print("Classification alone would flag all of these equally, ranking tells me which ones to look at first.")


16262 pages are labeled 'declining' out of 30000 total.
Classification alone would flag all of these equally, ranking tells me which ones to look at first.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

What I'm predicting: is_declining_label, which just means trend_direction == "down".

This isn't something directly observed like a click or a sale, it's a label
someone defined based on trend_pct. So it's a proxy, not ground truth. That's
worth keeping in mind, because if the definition of "declining" changes later,
this label would change too. I'll be careful to say "pages flagged as
declining under this definition" rather than acting like it's a fixed fact.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df_check["is_declining_label"] = (df_check["trend_direction"] == "down").astype(int)
print(df_check[["trend_direction", "trend_pct", "is_declining_label"]].head(5))

  trend_direction  trend_pct  is_declining_label
0            down      -41.4                   1
1            down      -57.7                   1
2            down      -60.9                   1
3          stable      -13.8                   0
4            down      -34.7                   1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

I'm using Precision@50 as my metric.

Basically: out of the top 50 pages my model ranks as most urgent, how many
are actually declining? I picked this because it mirrors how someone would
actually use this, a content team isn't going to review thousands of pages,
they'll work through a realistic batch. So precision at that batch size tells
me something useful, unlike overall accuracy, which would look artificially
high just because most pages aren't declining anyway.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

declining_rate = df_check["is_declining_label"].mean()
print(f"Share of pages that are declining: {declining_rate:.3f}")
print("A model that just guesses 'not declining' for everyone would still score high on accuracy -")
print("that's why Precision@50 makes more sense here, since it actually checks the top-ranked pages.")

Share of pages that are declining: 0.542
A model that just guesses 'not declining' for everyone would still score high on accuracy -
that's why Precision@50 makes more sense here, since it actually checks the top-ranked pages.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

My unit of analysis: one row = one content page. Below I load the data and
show a slice of the columns that matter for my lane, impressions, position,
CTR, and whether the page is trending up or down.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{df.shape[0]} rows, {df.shape[1]} columns")
df[["content_id", "impressions_90d", "avg_position", "ctr", "trend_direction"]].head(5)

30000 rows, 44 columns


,content_id,impressions_90d,avg_position,ctr,trend_direction
0,content_304f48230142,3803,10.6,0.76,down
1,content_a1fb4e703a9e,15320,20.3,0.05,down
2,content_9aa793d4d895,12581,36.5,0.09,down
3,content_331d6c4de07b,11751,6.2,0.49,stable
4,content_d99b7a2d90ca,19140,44.0,0.13,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Why ML beats a fixed rule: I actually tested this directly in the last two
notebooks. My hand-written rule (stale pages that are still getting
impressions) did fine at Precision@50, and in one test it even beat a simple
decision tree once I checked it on data the tree hadn't seen before (0.680 vs
0.660). But that's exactly the problem with fixed rules, they can hold up
okay at the very top of a ranked list, but they run out of signal further
down, since they're only checking one or two conditions I picked ahead of
time. A model can pick up on softer combinations of features (like word
count, position, and CTR together) that I wouldn't think to hardcode as an
if/else.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Hand rule Precision@50 (held-out): 0.680")
print("Tree      Precision@50 (held-out): 0.660")
print("Even a simple rule can hold its own — but it gets fragile once you look past the very top of the list.")

Hand rule Precision@50 (held-out): 0.680
Tree      Precision@50 (held-out): 0.660
Even a simple rule can hold its own — but it gets fragile once you look past the very top of the list.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.